In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

print(train.shape)

Mounted at /content/drive
(975, 30)


In [2]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator

In [3]:
def create_validation_copy(df, seed=42):

    rng = np.random.default_rng(seed)

    df_copy = df.copy()

    hidden_positions = []

    option_cols = df.columns[2:]

    for row_idx in range(len(df)):

        row = df.loc[row_idx, option_cols]

        known = np.where(~row.isna())[0]

        if len(known) < 10:
            continue

        hide = rng.choice(
            known,
            size=max(1, int(len(known)*0.2)),
            replace=False
        )

        for pos in hide:

            hidden_positions.append(
                (
                    row_idx,
                    option_cols[pos],
                    row.iloc[pos]
                )
            )

            df_copy.loc[row_idx, option_cols[pos]] = np.nan

    return df_copy, hidden_positions

In [4]:
val_df, hidden_positions = create_validation_copy(train)

print(len(hidden_positions))

3958


In [5]:
def evaluate_linear(val_df, hidden_positions):

    option_cols = val_df.columns[2:]

    preds = []

    truths = []

    for row_idx, col, true_val in hidden_positions:

        row = val_df.loc[row_idx, option_cols].values.astype(float)

        x = np.arange(len(row))

        valid = ~np.isnan(row)

        if valid.sum() < 2:
            continue

        f = interp1d(
            x[valid],
            row[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        target_idx = option_cols.get_loc(col)

        pred = float(f(target_idx))

        preds.append(pred)
        truths.append(true_val)

    mse = np.mean(
        (np.array(preds) - np.array(truths))**2
    )

    return mse

In [6]:
linear_mse = evaluate_linear(
    val_df,
    hidden_positions
)

print("Linear MSE =", linear_mse)

Linear MSE = 0.00018914326083231846


In [7]:
def evaluate_pchip(val_df, hidden_positions):

    option_cols = val_df.columns[2:]

    preds = []

    truths = []

    for row_idx, col, true_val in hidden_positions:

        row = val_df.loc[row_idx, option_cols].values.astype(float)

        x = np.arange(len(row))

        valid = ~np.isnan(row)

        if valid.sum() < 3:
            continue

        try:

            f = PchipInterpolator(
                x[valid],
                row[valid]
            )

            target_idx = option_cols.get_loc(col)

            pred = float(f(target_idx))

            preds.append(pred)
            truths.append(true_val)

        except:
            pass

    mse = np.mean(
        (np.array(preds) - np.array(truths))**2
    )

    return mse

In [8]:
pchip_mse = evaluate_pchip(
    val_df,
    hidden_positions
)

print("PCHIP MSE =", pchip_mse)

PCHIP MSE = 0.00021550208227338766


In [9]:
def evaluate_quadratic(val_df, hidden_positions):

    option_cols = val_df.columns[2:]

    preds = []

    truths = []

    for row_idx, col, true_val in hidden_positions:

        row = val_df.loc[row_idx, option_cols].values.astype(float)

        x = np.arange(len(row))

        valid = ~np.isnan(row)

        if valid.sum() < 4:
            continue

        try:

            f = interp1d(
                x[valid],
                row[valid],
                kind="quadratic",
                fill_value="extrapolate"
            )

            target_idx = option_cols.get_loc(col)

            pred = float(f(target_idx))

            preds.append(pred)
            truths.append(true_val)

        except:
            pass

    mse = np.mean(
        (np.array(preds) - np.array(truths))**2
    )

    return mse

In [10]:
quadratic_mse = evaluate_quadratic(
    val_df,
    hidden_positions
)

print("Quadratic MSE =", quadratic_mse)

Quadratic MSE = 0.0002748432546221621
